In [0]:
df = spark.sql("select * from project.silver.customers")
display(df)

In [0]:
df = df.dropDuplicates(subset=['customer_id'])


In [0]:
# dbutils.widgets.text('innit_load_flag','')
# innit_load_flag = int(dbutils.widgets.get('innit_load_flag'))

### Dividing New and Old Records

In [0]:
# if innit_load_flag == 0:
# OR
if spark.catalog.tableExists('project.gold.customers'):
  df_old = spark.sql("select DimKeyCustomer, customer_id, create_date, update_date from project.gold.customers")


else:
  df_old = spark.sql("select 0 DimKeyCustomer, 0 customer_id, 0 create_date, 0 update_date from project.silver.customers where 1 = 0")
    

In [0]:
df_old.display()

### Renaming Columns of old_df

In [0]:
df_old = df_old.withColumnRenamed('dimKeyCustomer', 'old_dimKeyCustomer') \
              .withColumnRenamed('customer_id', 'old_customer_id') \
              .withColumnRenamed('create_date', 'old_create_date') \
              .withColumnRenamed('update_date', 'old_update_date')  

### Applying Joins with the Old Records

In [0]:
df_join = df.join(df_old, df.customer_id == df_old.old_customer_id, 'left')

In [0]:
df_join.display()

### Separating Old VS New Records

In [0]:
df_new = df_join.filter(df_join['old_dimkeyCustomer'].isNull())
display(df_new)

In [0]:
df_old = df_join.filter(df_join['old_dimkeyCustomer'].isNotNull())
display(df_old.limit(10))

### Preparing Df_old

In [0]:
from pyspark.sql.functions import to_timestamp,current_timestamp

In [0]:
# Dropping all the columsn that are not required
df_old = df_old.drop('old_customer_id', 'old_update_date')

#  Renaming "old_dimKeyCustomer" to "dimKeyCustomer"
df_old = df_old.withColumnRenamed('old_dimKeyCustomer', 'dimKeyCustomer')

#  Renaming "old_create_date" to "create_date"
df_old = df_old.withColumnRenamed("old_create_date", "create_date")
df_old = df_old.withColumn("create_date", to_timestamp("create_date"))

#  Recreating "update_date" column with Current timestamp
df_old = df_old.withColumn("update_date", current_timestamp()) 

In [0]:
df_old.display()

### Preparing df_new

In [0]:
# Dropping all the columsn that are not required
df_new = df_new.drop('old_dimKeyCustomer', 'old_customer_id', 'old_update_date', 'old_create_date')

#  Recreating "iupdate_date" & "created_date" with Current Timestamp
df_new = df_new.withColumn("create_date", current_timestamp())
df_new = df_new.withColumn("update_date", current_timestamp()) 

In [0]:
display(df_new.limit(10))

###  Surrogate Key - From 1

In [0]:
from pyspark.sql.functions import *

In [0]:
df_new = df_new.withColumn('dimKeyCustomer',monotonically_increasing_id() +lit(1))
display(df_new)

### Adding Max Surrogate Key

In [0]:
if spark.catalog.tableExists('project.gold.customers'):
    df_max_surr = spark.sql("SELECT MAX(dimKeyCustomer) as max_surrogate_key FROM project.gold.customers")
    # Converting df_max_surr to Surrogate Key Variable
    max_surrogate_key = df_max_surr.collect()[0]['max_surrogate_key']
else:
    max_surrogate_key = 0

In [0]:
df_new = df_new.withColumn('dimKeyCustomer',lit(max_surrogate_key) + col('dimKeyCustomer'))

In [0]:
df_new.display()

### Union of df_old & df_new

In [0]:
df_final = df_new.unionByName(df_old)

In [0]:
df_final.display()

### SCD TYPE 1


In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("project.gold.customers"):
    dlt_obj = DeltaTable.forName(spark, "project.gold.customers")
    dlt_obj.alias('target').merge(df_final.alias('source'),'target.dimKeyCustomer = source.dimKeyCustomer') \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()    
else:
    df_final.write.mode("overwrite").saveAsTable("project.gold.customers") 
    

In [0]:
%sql
SELECT * FROM project.gold.customers